# 🎵 MiniMax Music 3 - Google Colab (GPU T4 16GB / L4 / A100)
Chạy trọn gói Inference Engine, REST API Gateway và Giao diện Web UI trên Google Colab GPU miễn phí.

In [ ]:
#@title 1. Kiểm tra GPU & Cài đặt Môi trường (1-Click Setup)
!nvidia-smi
!git clone https://github.com/hoainamgo/minimax-music3.git /content/minimax-music3
%cd /content/minimax-music3
!bash lightning_setup.sh

In [ ]:
#@title 2. Khởi chạy Inference Engine GPU (Chạy nền cổng 8086)
import subprocess
import time

engine_process = subprocess.Popen(
    ["./runtime/mm-server", "--models", "models", "--host", "127.0.0.1", "--port", "8086", "--keep-loaded", "--max-batch", "1"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("Đang khởi động MiniMax Music 3 Engine...")
time.sleep(5)
!curl -s http://127.0.0.1:8086/health

In [ ]:
#@title 3. Khởi chạy REST API Gateway & Cloudflare Tunnel (apimusic.ksmart.com.es)
#@markdown Nhập Cloudflare Tunnel Token (hoặc để trống để tạo link tạm Quick Tunnel):
CLOUDFLARE_TOKEN = "" #@param {type:"string"}

import subprocess
import os

# Bật FastAPI Gateway
api_process = subprocess.Popen(["python", "api_server.py"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

# Cài đặt cloudflared
!curl -s -L --output cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared.deb > /dev/null 2>&1

if CLOUDFLARE_TOKEN.strip():
    print("Đang kết nối tới apimusic.ksmart.com.es qua Tunnel Token...")
    !cloudflared tunnel run --token {CLOUDFLARE_TOKEN}
else:
    print("Đang tạo Quick Tunnel miễn phí...")
    !cloudflared tunnel --url http://127.0.0.1:8000

In [ ]:
#@title 4. Hoặc Chạy Trực Tiếp CLI Tạo Nhạc trong Colab
!python music_cli.py generate --title "Colab Melody" --style "Modern pop ballad, emotional piano, warm strings" --vocal female --duration 90